<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/assignments/Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


## 1. Visualizing the Mechanics

In the 2PL model, the difficulty parameter $b_i$ determines the horizontal location of the Item Response Function (IRF), while the discrimination parameter $a_i$ determines its steepness. 

When you increase $b_i$, the curve shifts horizontally to the **right**. This means a user requires a higher latent ability $\theta$ to have a 50% chance of answering the item correctly, indicating a more difficult question.

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Define theta range
theta = np.linspace(-4, 4, 400)

# 2PL function
def p_2pl(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

fig = go.Figure()

# Curve 1, 2, 3: Same discrimination (a=1.5), different difficulties
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 1.5, -1), name="a=1.5, b=-1 (Easy)"))
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 1.5, 0), name="a=1.5, b=0 (Medium)"))
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 1.5, 1), name="a=1.5, b=1 (Hard)"))

# Curve 4: Different discrimination (a=0.5), medium difficulty
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 0.5, 0), 
                         name="a=0.5, b=0 (Low Discrimination)", 
                         line=dict(dash='dash')))

fig.update_layout(title="2PL Item Response Functions",
                  xaxis_title="Latent Ability (theta)",
                  yaxis_title="Probability of Correct Response P(Y=1)",
                  template="plotly_white")
fig.show()

 

## 2. Sequential Likelihood Contribution

Because the item responses are assumed to be conditionally independent given the latent ability $\theta$, the likelihood of a single response follows a Bernoulli distribution. 

The likelihood contribution of a single new response $y_k$ at step $k$ is:

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

Substituting the 2PL model formula, this becomes:

$$L(y_k \mid \theta) = \left( \frac{1}{1+e^{-a_k(\theta-b_k)}} \right)^{y_k} \left( \frac{e^{-a_k(\theta-b_k)}}{1+e^{-a_k(\theta-b_k)}} \right)^{1-y_k}$$

The joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is the product of the individual likelihoods up to step $k$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^{k} L(y_j \mid \theta)$$

## 3. Mathematical Formulation of the Running Update

By Bayes' theorem, the posterior density at step $k$ is proportional to the product of the likelihood of the new observation and the prior density (which is the posterior from step $k-1$).

The recursive relationship is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \times f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

To formulate the exact probability density function, normalize by dividing by the marginal likelihood:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \times f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{-\infty}^{\infty} L(y_k \mid u) \times f_{\Theta \mid \mathbf{Y}^{(k-1)}}(u \mid \mathbf{y}^{(k-1)}) \, du}$$

## 4. Dynamic Shifting

If a user answers a highly difficult item correctly ($y_k = 1$ and $b_k$ is large), the likelihood function $L(y_k=1 \mid \theta)$ evaluates near zero for low $\theta$ values and approaches 1 for $\theta$ values greater than $b_k$. 

Multiplying the previous posterior density by this right-leaning likelihood function heavily penalizes the probability mass at the lower end of the ability scale and preserves it at the higher end. Consequently, the peak of the new running posterior density distribution shifts mathematically to the right toward a higher ability estimate.

## 5. Tracking Certainty and Sharpness

The discrimination parameter $a_k$ determines how steeply the response probability transitions from 0 to 1. 

*   **When $a_k$ is very large:** The likelihood function acts almost like a step function. It drastically slashes the probability mass on one side of $b_k$ while preserving the other side. This aggressive filtering rapidly reduces the variance of the posterior distribution, resulting in a sharper density and increasing certainty about the user's ability.
*   **When $a_k$ is very small:** The likelihood function is flat and uninformative. Multiplying the previous posterior by a nearly flat curve minimally affects its shape. The variance remains largely unaffected, indicating little certainty was gained from that interaction.

## 6. Numerical Implementation of a Running Grid

Because the exact posterior lacks a closed-form solution (it is not conjugate to the standard normal prior), it must be approximated numerically over a discrete grid.

1.  **Define the Grid:** Create a linearly spaced array of $\theta$ values (e.g., from -4 to 4 with 1000 points). Let the distance between points be $\Delta\theta$.
2.  **Initialize Prior:** Calculate the standard normal density for each point on the grid. This array is the starting posterior $k=0$.
3.  **Update Loop:** When a new response $y_k$ is observed:
    *   Calculate the likelihood array for every point on the grid using $a_k$, $b_k$, and $y_k$.
    *   Perform element-wise multiplication of the likelihood array and the current posterior array to compute the unnormalized posterior.
    *   **Sequential Normalization:** Calculate the area under the unnormalized curve by summing all elements and multiplying by $\Delta\theta$. Divide the unnormalized posterior array by this area to ensure it integrates to 1. 
4.  **Extract Estimates:** Find the mean (sum of $\theta \times \text{posterior} \times \Delta\theta$) and MAP (the $\theta$ value where the posterior array is at its maximum).

## 7. Evaluating Convergence over the Timeline


In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# 1. Setup Simulation Parameters
np.random.seed(42)
n_items = 20
theta_true = 0.75

# Generate item parameters
b_params = np.random.normal(0, 1, n_items)
a_params = np.random.uniform(0.5, 2.0, n_items)

# Setup numerical grid for theta
grid_size = 1000
theta_grid = np.linspace(-4, 4, grid_size)
delta_theta = theta_grid[1] - theta_grid[0]

# 2PL probability function
def prob_correct(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# 2. Tracking Variables
posterior = norm.pdf(theta_grid) # Initial prior: N(0,1)
map_estimates = [0.0]            # Prior MAP
mean_estimates = [0.0]           # Prior Mean

# 3. Simulate and Update
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]
    
    # Simulate user response
    p_true = prob_correct(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0
    
    # Calculate Likelihood for the grid
    p_grid = prob_correct(theta_grid, a_k, b_k)
    likelihood = (p_grid ** y_k) * ((1 - p_grid) ** (1 - y_k))
    
    # Update and Normalize Posterior
    unnormalized_posterior = posterior * likelihood
    normalization_constant = np.sum(unnormalized_posterior) * delta_theta
    posterior = unnormalized_posterior / normalization_constant
    
    # Extract MAP and Mean
    map_est = theta_grid[np.argmax(posterior)]
    mean_est = np.sum(theta_grid * posterior * delta_theta)
    
    map_estimates.append(map_est)
    mean_estimates.append(mean_est)

# 4. Visualize with Plotly
steps = list(range(n_items + 1))

fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=mean_estimates, mode='lines+markers', name="Posterior Mean"))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name="MAP Estimate"))
fig.add_trace(go.Scatter(x=steps, y=[theta_true]*(n_items + 1), mode='lines', 
                         name="True Theta (0.75)", line=dict(dash='dash', color='red')))

fig.update_layout(title="Convergence of Sequential Bayesian Estimates",
                  xaxis_title="Item Number (k)",
                  yaxis_title="Latent Ability Estimate",
                  template="plotly_white")
fig.show()

### Analysis of Convergence

As $k$ increases, the distance between both estimators (Posterior Mean and MAP) and $\theta_{\text{true}}$ decreases, despite minor fluctuations caused by the randomness of individual responses. 

Each observation contributes new likelihood information that multiplies against the prior. As items accumulate, the product of these likelihoods overrides the standard normal prior. The running posterior distribution narrows and centers increasingly closer to the true parameter of 0.75. This behavior mathematically reflects the platform's growing measurement confidence, transitioning from an uncertain prior to a highly constrained, data-driven posterior.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

## 1. Structural Probability and Properties

The Beta distribution is defined by two shape parameters, $\alpha$ and $\beta$, which act as "pseudo-counts" of successes and failures.

*   **Uninformative state $(\alpha=1, \beta=1)$:** The density is perfectly flat (a Uniform distribution). The center of mass is exactly at 0.5, representing complete uncertainty where all values of $\theta$ are equally likely.
*   **Right-skewed state $(\alpha=2, \beta=8)$:** The parameter $\beta$ is dominant, pulling the probability mass strongly toward 0. The distribution has a long tail to the right (right-skewed), indicating a high belief that the true probability is low.
*   **Left-skewed state $(\alpha=8, \beta=2)$:** The parameter $\alpha$ is dominant, pulling the probability mass strongly toward 1. The distribution has a long tail to the left (left-skewed), indicating a high belief that the true probability is high.


In [3]:

import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta = np.linspace(0, 1, 500)

fig = go.Figure()

# (1, 1) Uninformative
fig.add_trace(go.Scatter(x=theta, y=beta.pdf(theta, 1, 1), 
                         name="alpha=1, beta=1 (Uniform)", line=dict(width=3)))

# (2, 8) Right-skewed
fig.add_trace(go.Scatter(x=theta, y=beta.pdf(theta, 2, 8), 
                         name="alpha=2, beta=8 (Right-skewed)", line=dict(width=3)))

# (8, 2) Left-skewed
fig.add_trace(go.Scatter(x=theta, y=beta.pdf(theta, 8, 2), 
                         name="alpha=8, beta=2 (Left-skewed)", line=dict(width=3)))

fig.update_layout(title="Beta Distribution Probability Density Functions",
                  xaxis_title="Conversion Rate (theta)",
                  yaxis_title="Density",
                  template="plotly_white")
fig.show()



## 2. Sequential Likelihood and Joint History

Because each impression is an independent Bernoulli trial, the likelihood of a single new response $y_k$ given $\theta$ is:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

The joint likelihood for the running history vector $\mathbf{y}^{(k)}$ is the product of the individual likelihoods from step 1 to step $k$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^{k} \theta^{y_j} (1 - \theta)^{1 - y_j} = \theta^{\sum_{j=1}^k y_j} (1 - \theta)^{k - \sum_{j=1}^k y_j}$$

## 3. Closed-Form Analytical Updates (Conjugacy)

Using Bayes' Theorem for a sequential update, the posterior at step $k$ is proportional to the product of the likelihood of $y_k$ and the prior (which is the posterior from step $k-1$):

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \times f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Substituting the known forms:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \times \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

By combining the exponents, we get:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

This resulting expression matches the exact functional form of a Beta distribution, proving that the Beta prior and Bernoulli likelihood are conjugate. The explicit closed-form parameter updates are simply:

$$\alpha_k = \alpha_{k-1} + y_k$$
$$\beta_k = \beta_{k-1} + (1 - y_k)$$

The Posterior Mean of the latent parameter $\Theta$ at time step $k$ evaluates directly to:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

## 4. Dynamic Shifting Mechanics

Because the updates are simple additions to the shape parameters:
*   An observed **click** ($y_k = 1$) increments $\alpha_k$ by 1, leaving $\beta_k$ unchanged. This mathematically shifts the distribution's center of mass to the right, toward a higher estimated conversion rate.
*   An observed **non-click** ($y_k = 0$) increments $\beta_k$ by 1, leaving $\alpha_k$ unchanged. This shifts the distribution's center of mass to the left, toward a lower estimated conversion rate.

**Contrast with Non-Conjugate Setups (e.g., 2PL IRT):** 
In this conjugate setup, updating our belief takes constant time via simple arithmetic on two parameters ($\alpha$ and $\beta$), maintaining an exact continuous functional form. In non-conjugate models like the 2PL IRT model, the posterior cannot be simplified algebraically. The platform must approximate the continuous distribution over a discrete grid, calculate the likelihood for every grid point, multiply arrays, and numerically integrate to compute a normalizing constant. This makes non-conjugate models significantly more computationally expensive to update per impression.

## 5. Running Point Estimators

Based on the updated parameters $\alpha_k$ and $\beta_k$ at step $k$, the exact closed-form point estimates are:

*   **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$):
    $$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

*   **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$):
    $$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad (\text{defined for } \alpha_k, \beta_k \ge 1, \text{ excluding } \alpha_k=\beta_k=1)$$

## 6. Performance Tracking and Convergence Analysis

As $k$ approaches $100$, both the Posterior Mean and the MAP estimate converge strongly toward the true underlying rate $\theta_{\text{true}} = 0.35$. In the earliest steps, the estimates oscillate wildly due to small sample sizes. However, as evidence accumulates (higher $k$), the denominator of the estimators ($\alpha_k + \beta_k$) grows larger, making each new single impression have a mathematically smaller marginal impact on the overall fraction. 

This implies that while the uninformative initial prior ($\alpha_0=1, \beta_0=1$) dictates the starting point, the likelihood of the observed data rapidly overwhelms it. The influence of the prior decays at a rate of $\frac{1}{k}$, allowing the objective data to dictate the system's belief as time progresses.

In [4]:
import numpy as np
import plotly.graph_objects as go

# 1. Setup Simulation Parameters
np.random.seed(101)
n_impressions = 100
theta_true = 0.35

# 2. Tracking Variables
alpha_k = 1
beta_k = 1
mean_estimates = [alpha_k / (alpha_k + beta_k)]
map_estimates = [np.nan] # MAP is undefined at (1,1)

# 3. Simulate and Update
for k in range(n_impressions):
    # Simulate user interaction
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    
    # Analytical Update (Conjugacy)
    alpha_k += y_k
    beta_k += (1 - y_k)
    
    # Calculate Estimators
    mean_est = alpha_k / (alpha_k + beta_k)
    
    # Handle MAP edge cases
    if (alpha_k + beta_k - 2) == 0:
        map_est = np.nan
    else:
        # Clamp to [0, 1] for edge cases early in the sequence
        map_est = max(0.0, min(1.0, (alpha_k - 1) / (alpha_k + beta_k - 2)))
        
    mean_estimates.append(mean_est)
    map_estimates.append(map_est)

# 4. Visualize with Plotly
steps = list(range(n_impressions + 1))

fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=mean_estimates, mode='lines', name="Posterior Mean", line=dict(width=2)))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines', name="MAP Estimate", line=dict(width=2, dash='dot')))
fig.add_trace(go.Scatter(x=steps, y=[theta_true]*(n_impressions + 1), mode='lines', 
                         name="True Theta (0.35)", line=dict(dash='dash', color='red', width=2)))

fig.update_layout(title="Sequential CTR Estimation Convergence (Beta-Bernoulli)",
                  xaxis_title="Impression Number (k)",
                  yaxis_title="Estimated CTR (theta)",
                  template="plotly_white")
fig.show()